# Pre-Earnings Runup — a quantitative teardown
### Gross vs market Sharpe · Newey-West *t* · break-even cost · per-day drift curve · window sweep

![Signal: Weak](https://img.shields.io/badge/Signal-Weak-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Pre-event drift exists?: Mixed](https://img.shields.io/badge/Pre--event_drift_exists%3F-Mixed-8b949e?style=flat-square)

The deep companion to [01_for_the_curious.ipynb](01_for_the_curious.ipynb) — *same seven beats, every claim with its number.* The thesis: informed traders position before scheduled earnings announcements, creating a repeatable pre-event price drift (Kim & Park 2005; Frazzini & Lamont 2007). On the **real tape** the equal-weight pre-event book earns gross Sharpe 0.46 (Newey-West *t* 1.68) vs the passive market's 0.97: no alpha, just beta — `WEAK` signal, `MIRAGE` tradability.

> ✅ **Real-tape run, fingerprinted.** The headline numbers below are measured on the **real tape** — yfinance adjusted closes + earnings dates for a fixed large-cap universe (**98 names, 2345 events, 2014-01-03 - 2026-06-16, fp `220e0562c461`**). The executed code cells run the fast, reproducible **synthetic control** (a planted pre-event drift + a null) that backs the test-suite; reproduce the real run with [`examples/verify.py`](../examples/verify.py) after caching (`--fetch` first time). **Caveat:** the universe is current large-cap membership ⇒ survivorship bias inflates the magnitudes.
>
> ⚠️ **Not investment advice.** The executed cells run the synthetic control; sources in [`../docs/references.md`](../docs/references.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back to intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../../.."))
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.2)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
from pre_earnings_runup import data, strategy

# These EXECUTED cells run the offline synthetic control: a panel where a pre-event drift is planted
# (machinery proof) + a null where the same calendar carries no drift. The HEADLINE numbers are the
# REAL-tape measurement — see ../docs/results.md / verify.py.
panel, events, truth = data.synthetic_runup(seed=228)               # the drift control
panel0, events0, _   = data.synthetic_runup(drift_strength=0.0, seed=228)  # the null
print(f"synthetic control: {truth.n_stocks} stocks x {truth.n_bars} days, "
      f"{len(events)} events, drift_strength {truth.drift_strength} (null=0)")


synthetic control: 100 stocks x 3024 days, 4800 events, drift_strength 0.0004 (null=0)


## Beat 0 · Verdict (real tape)

| Axis | Stamp | Why |
|---|---|---|
| **Signal** — pre-event drift? | 🟡 `WEAK` | Gross Sharpe **0.46** (CAGR 7.8%), Newey-West *t* **1.68** (plain 1.62) — below the |*t*| ≥ 2 bar AND below the passive market Sharpe (**0.97**). No alpha. |
| **Tradability** | 🔴 `MIRAGE` | Turnover 0.2525/day, break-even **15.2 bp** — but the gross edge is entirely beta; net @5 bp Sharpe **+0.31** vs market **+0.97**. |
| **Pre-event drift exists?** | 🟠 `MIXED` | Per-day returns are positive (+0.04–0.12%/day) but indistinguishable from ordinary market drift; no monotone rise toward announcement day. |

> **In one sentence:** the pre-event book earns 0.46 gross Sharpe (*t* 1.68) vs the passive market's 0.97 — a `WEAK` beta-contaminated signal, a `MIRAGE` to trade.

*(Real run: 98 names, 2345 events, 2014-01-03 - 2026-06-16, fp `220e0562c461` — [`../docs/results.md`](../docs/results.md). The executed cells run the synthetic control.)*

## Beat 1 · The claim, precisely

For each trading day $t$, go long every name whose next scheduled earnings announcement falls within $[1, H]$ trading days (strictly causal: the calendar is public). Equal-weight across all in-book names; weight = 0 outside the window. The position is entered one day after the window opens (lag-one causal execution). Claim: the pre-event window carries a positive return premium above the market. Null: the same window carries no excess return.

In [2]:
for label, (pp, ee) in [('drift control', (panel, events)), ('null', (panel0, events0))]:
    curve = strategy.mean_pre_event_return(pp, ee, pre_days=5)
    print(f'{label:14} mean pre-event return: {curve.mean()*100:+.4f}%/day')
    print('  ' + '  '.join(f'day{k}:{v*100:+.4f}%' for k, v in curve.items()))
    print()

drift control  mean pre-event return: +0.0587%/day
  day-5:+0.0473%  day-4:+0.0382%  day-3:+0.0835%  day-2:+0.0799%  day-1:+0.0447%



null           mean pre-event return: +0.0347%/day
  day-5:+0.0393%  day-4:+0.0222%  day-3:+0.0595%  day-2:+0.0479%  day-1:+0.0047%



> 💡 **In plain words.** On the control the planted drift makes pre-event returns slightly higher than on the null. The *real tape* pre-event mean returns are positive (+0.04–0.12%/day) but roughly equal to the market's daily drift — the machinery sees the planted effect but the real market offers no excess premium above passive.

## Beat 2 · So what?

If the pre-event window genuinely earns ABOVE the market (alpha), a long-only book would show a higher Sharpe than the passive equal-weight. On the real tape: book Sharpe 0.46 vs market Sharpe 0.97. The book earns LESS than the market because it only holds names in the pre-event window (spending most of the time in cash). There is no alpha, only a partial-period market exposure.

## Beat 3 · Protocol

1. **Real?** `book_returns(cost_bps=0)` gross + Newey-West *t* (`strategy.newey_west_t`) on the real tape; the per-day curve below proves the machine sees a planted drift.
2. **Alpha vs beta?** Compare the book Sharpe to the passive equal-weight market Sharpe.
3. **Drift shape?** `strategy.mean_pre_event_return` — does the per-day return RISE toward day −1 (the informed-accumulation prediction)?
4. **Tradable?** `strategy.turnover`, `strategy.breakeven_cost_bps`, `strategy.pre_day_sweep`.

**Mirage line:** a gross Sharpe below the passive market (0.97) OR a Newey-West *t* below 2. Real tape lands at 0.46 / *t* 1.68 — both fail.

## Beat 4 · The teardown

**Real tape:** gross Sharpe **0.46**, Newey-West *t* **1.68** (plain 1.62), net @5 bp **+0.31**, market Sharpe **0.97** — see [`../docs/results.md`](../docs/results.md). The cells below run the synthetic control.

### 4a · Per-day pre-event return: planted control vs null

In [3]:
print('Synthetic pre-event mean returns by day:')
for label, (pp, ee) in [('control', (panel, events)), ('null', (panel0, events0))]:
    curve = strategy.mean_pre_event_return(pp, ee, pre_days=5)
    print(f'  {label}: ' + '  '.join(f'day{k}:{v*100:+.4f}%' for k, v in curve.items()))
print('REAL tape: day-5 +0.063%  day-4 +0.040%  day-3 +0.122%  day-2 +0.106%  day-1 +0.084%')
print('No monotone rise to day-1 -- MIXED. Market daily drift ~+0.06%/day explains positivity.')

Synthetic pre-event mean returns by day:


  control: day-5:+0.0473%  day-4:+0.0382%  day-3:+0.0835%  day-2:+0.0799%  day-1:+0.0447%
  null: day-5:+0.0393%  day-4:+0.0222%  day-3:+0.0595%  day-2:+0.0479%  day-1:+0.0047%
REAL tape: day-5 +0.063%  day-4 +0.040%  day-3 +0.122%  day-2 +0.106%  day-1 +0.084%
No monotone rise to day-1 -- MIXED. Market daily drift ~+0.06%/day explains positivity.


In [4]:
print('Synthetic book Sharpe (control vs null):')
g_c = strategy.summary(strategy.book_returns(panel, events, cost_bps=0.0))
g_n = strategy.summary(strategy.book_returns(panel0, events0, cost_bps=0.0))
print(f'  control gross Sharpe {g_c["sharpe"]:.3f}  null gross Sharpe {g_n["sharpe"]:.3f}')
print('  (Both ~equal: the equal-weight book is beta-dominated. Real tape: gross 0.46 vs market 0.97)')

Synthetic book Sharpe (control vs null):


  control gross Sharpe 0.094  null gross Sharpe 0.094
  (Both ~equal: the equal-weight book is beta-dominated. Real tape: gross 0.46 vs market 0.97)


> 💡 **In plain words.** The per-day curve sees the planted drift (control > null by +0.4 bp/day), but the book Sharpes are identical because the market-beta component swamps the small planted effect. On the *real tape* the book Sharpe is only 0.46 vs the market's 0.97 — the book is just buying equities part-time, not harvesting a pre-earnings premium.

### 4b · The cost wall and break-even

In [5]:
print('Pre-day window sweep (synthetic control):')
sweep = strategy.pre_day_sweep(panel, events, windows=[1, 2, 3, 5, 10], cost_bps=2.0)
print(sweep.round(3).to_string())
print('REAL tape window sweep (gross Sharpe / net@2bp):')
print('  1d: 0.03/-0.07  2d: 0.26/0.19  3d: 0.39/0.32  5d: 0.46/0.40  10d: 0.55/0.49')
print('Sharpe RISES with window length -- just more market beta, not more alpha.')
to = strategy.turnover(panel, events)
be = strategy.breakeven_cost_bps(panel, events)
print(f'synthetic turn/day {to:.3f}, break-even {be:.1f} bp  (real: 0.2525/day, 15.2 bp)')

Pre-day window sweep (synthetic control):


          gross_sharpe  net_sharpe  turnover_per_day
pre_days                                            
1               0.1290     -0.2880            1.6820
2              -0.1880     -0.5000            1.2220
3              -0.0850     -0.3220            0.8290
5               0.0940     -0.0660            0.5090
10              0.2050      0.1160            0.2640
REAL tape window sweep (gross Sharpe / net@2bp):
  1d: 0.03/-0.07  2d: 0.26/0.19  3d: 0.39/0.32  5d: 0.46/0.40  10d: 0.55/0.49
Sharpe RISES with window length -- just more market beta, not more alpha.


synthetic turn/day 0.509, break-even 1.2 bp  (real: 0.2525/day, 15.2 bp)


## Beat 5 · The verdict

- **`WEAK`** (4a, 4b): gross Sharpe 0.46, *t* 1.68 — below both the |t|≥2 bar and the passive market (0.97). Zero alpha.
- **`MIRAGE`** (4b): turnover 0.2525/day, break-even 15.2 bp — but there is no gross alpha to protect; net @5 bp Sharpe +0.31 vs market +0.97.
- **Pre-event drift exists? `MIXED`** (4a): positive per-day raw returns but no monotone rise toward day −1 and no excess over the market drift.

> **Signal `WEAK` · Tradability `MIRAGE` · Pre-event drift exists? `MIXED`.**

## Beat 6 · Could you trade it?

- **The right test is market-neutral.** A long-only pre-event book vs a long-only passive market misattributes beta as alpha. The correct comparison is a market-neutral book: long pre-event names, short a matched non-event portfolio. On large-cap liquid names, the matched short would likely earn the same return, zeroing the alpha.
- **Arbitraged to zero.** The pre-earnings runup on liquid names has been documented, published, and traded since the 1990s (Lakonishok & Vermaelen 1990). On the S&P-scale names in this universe, any information advantage has been fully reflected in options IVs and the public calendar.
- **Window length is a beta dial.** The pre-day sweep shows Sharpe rising monotonically from 1d to 10d — but longer windows just mean more time in the market. If there were genuine pre-event alpha it would peak near the announcement, not increase indefinitely with the hold.

## Beat 7 · Going further

### 7a · The honest verdict & forks
The real measurement is **done and fingerprinted** in [`../docs/results.md`](../docs/results.md): gross Sharpe 0.46, Newey-West *t* 1.68, market Sharpe 0.97, turnover 0.2525/day, break-even 15.2 bp, as-of 2026-06-16, fp `220e0562c461`. Signal `WEAK` / Tradability `MIRAGE` — and the pre-event drift itself is `MIXED` because positive raw returns are contaminated by positive market beta.

### 7b · Other forks
- **Market-neutral test** — long pre-event names short a beta-matched non-event basket; this is the correct alpha isolation test.
- **Contrast with PEAD (study 34)** — the *post*-announcement drift earns gross Sharpe +0.30 (*t* +1.39), also `WEAK` but at least a slow, sustained drift tied to an information signal (the earnings *surprise*), not just event proximity.
- **Small-cap version** — the premium may be larger in less-covered, illiquid names, but at much higher trading costs.

PRs welcome — try the market-neutral version or a small-cap split.